# 🏓 Pickleball Base Vision AI — Player Detector Colab A100 Training (YOLO26)

This notebook trains/fine-tunes the **Pickleball Player Detector** using Ultralytics' newest **YOLO26 (NMS-Free End-to-End)** architecture on the isolated 5,398-image player dataset on an **NVIDIA A100 GPU**.

### Included Dataset:
- **Isolated Player Split**: 5,398 images (4,988 train / 217 val / 193 test), single class `0 = player`
- **Architecture**: `yolo26n.pt` (Nano) or `yolo26s.pt` (Small)
- **Input Resolution**: $640\times 640$ (standard) or $960\times 960$ (high-detail far court)

### Runtime Setup
1. Go to **Runtime** > **Change runtime type**
2. Hardware accelerator: **GPU** (A100 recommended)
3. Click **Save**

In [ ]:
# Step 1: Verify GPU Runtime
!nvidia-smi

In [ ]:
# Step 2: Install Ultralytics
!pip install -q ultralytics

In [ ]:
# Step 3: Upload and Extract players_clean_dataset.zip
# If using Google Drive:
# from google.colab import drive
# drive.mount('/content/drive')
# !unzip -q /content/drive/MyDrive/players_clean_dataset.zip -d ./dataset

# Or upload directly:
from google.colab import files
import os
if not os.path.exists("players_clean_dataset.zip"):
    print("Upload players_clean_dataset.zip from your machine:")
    uploaded = files.upload()

!unzip -q -o players_clean_dataset.zip -d ./
print("Dataset unzipped successfully!")

In [ ]:
# Step 4: Launch Training on YOLO26 Nano (NMS-Free End-to-End)
from ultralytics import YOLO

model = YOLO("yolo26n.pt")

train_results = model.train(
    data="players_clean/data.yaml",
    imgsz=640,
    epochs=40,
    patience=10,
    batch=64,
    workers=8,
    device=0,
    amp=True,
    optimizer="AdamW",
    lr0=0.001,
    weight_decay=0.0005,
    name="player_yolo26_colab_a100",
    save=True,
    plots=True,
    verbose=True,
)

In [ ]:
# Step 5: Evaluate on Held-Out Test Split (193 images)
test_metrics = model.val(
    data="players_clean/data.yaml",
    split="test",
    imgsz=640,
    device=0
)
print("Test Precision:", test_metrics.results_dict.get("metrics/precision(B)"))
print("Test Recall   :", test_metrics.results_dict.get("metrics/recall(B)"))
print("Test mAP@50   :", test_metrics.results_dict.get("metrics/mAP50(B)"))
print("Test mAP@50-95:", test_metrics.results_dict.get("metrics/mAP50-95(B)"))

In [ ]:
# Step 6: Download Best YOLO26 Player Checkpoint
import shutil
best_path = "runs/detect/player_yolo26_colab_a100/weights/best.pt"
if os.path.exists(best_path):
    shutil.copyfile(best_path, "player_yolo26_best.pt")
    files.download("player_yolo26_best.pt")
    print("Downloading player_yolo26_best.pt to your machine...")